In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds
from scipy.stats import spearmanr

# Seed para reprodutibilidade dos testes de permutação
RNG = np.random.default_rng(42)

In [ ]:
jobs    = pd.read_parquet('db/jobs.parquet')
ratings = pd.read_parquet('db/ratings.parquet')
movies  = pd.read_parquet('db/movies.parquet')
actors  = pd.read_parquet('db/actors.parquet')

In [ ]:
# Monta o df base: filme × ator com weight = 1/ordering
df = movies.merge(ratings, on='tconst', how='inner')
jobs.reset_index(inplace=True)
df = df.merge(jobs, on='tconst', how='inner')

df.set_index(['tconst', 'nconst'], inplace=True)

df.drop(columns=['genres', 'primaryTitle', 'originalTitle', 'startYear'], inplace=True)
df['weight'] = 1 / df['ordering']
df.drop(columns=['ordering', 'numVotes'], inplace=True)

df.reset_index(inplace=True)
df.head()

## Construção da matriz esparsa filme × ator

Cada linha representa um **filme** (`tconst`) e cada coluna um **ator** (`nconst`).  
O valor na célula $(i, j)$ é o `weight = 1/ordering` do ator $j$ no filme $i$ (0 caso não participe).

Com esta estrutura, cada filme vira um vetor no espaço de atores, e os componentes principais capturam padrões de co-participação ponderada.

In [ ]:
# Filtra atores que aparecem em pelo menos 50 filmes (reduz ruído e dimensionalidade)
MIN_FILMES_ATOR = 50
contagem_ator = df.groupby('nconst')['tconst'].nunique()
atores_freq = contagem_ator[contagem_ator >= MIN_FILMES_ATOR].index

df_filt = df[df['nconst'].isin(atores_freq)].copy()

print(f"Atores com >= {MIN_FILMES_ATOR} filmes: {len(atores_freq):,}")
print(f"Pares (filme, ator) após filtro:     {len(df_filt):,}")

In [ ]:
# Índices inteiros para filmes e atores
filmes = df_filt['tconst'].unique()
atores = df_filt['nconst'].unique()

filme_idx = {t: i for i, t in enumerate(filmes)}
ator_idx  = {n: j for j, n in enumerate(atores)}

rows = df_filt['tconst'].map(filme_idx).values
cols = df_filt['nconst'].map(ator_idx).values
vals = df_filt['weight'].values.astype(np.float32)

# Matriz esparsa filme × ator
M = csr_matrix((vals, (rows, cols)), shape=(len(filmes), len(atores)), dtype=np.float32)

print(f"Shape da matriz: {M.shape}  (filmes × atores)")
print(f"Esparsidade: {100 * (1 - M.nnz / (M.shape[0] * M.shape[1])):.2f}%")

## Centralização eficiente (PCA = SVD da matriz centrada)

PCA exige que cada coluna tenha média zero.  
Para matrizes esparsas, subtrair a média coluna a coluna densificaria tudo.  
A solução eficiente é subtrair a média implicitamente via produto matricial:  
se $\mu$ é o vetor de médias por coluna, então $Mv \approx Mv - \mu (\mathbf{1}^\top v)$ sem montar $M - \mathbf{1}\mu^\top$.

In [ ]:
# Média por coluna (ator) — shape (n_atores,)
col_mean = np.asarray(M.mean(axis=0)).ravel()  # float64 suficiente aqui

# LinearOperator que aplica A = M - 1*col_mean^T sem densificar
from scipy.sparse.linalg import LinearOperator

n_filmes, n_atores = M.shape

def matvec(v):
    """Computa (M - 1*mu^T) @ v"""
    return M @ v - col_mean.dot(v) * np.ones(n_filmes)

def rmatvec(v):
    """Computa (M - 1*mu^T)^T @ v"""
    return M.T @ v - col_mean * v.sum()

A_centered = LinearOperator(
    shape=(n_filmes, n_atores),
    matvec=matvec,
    rmatvec=rmatvec,
    dtype=np.float64
)

print("LinearOperator criado — centralização implícita pronta.")

In [ ]:
K = 50  # número de componentes principais

U, S, Vt = svds(A_centered, k=K)

# svds retorna valores singulares em ordem crescente — reordenar para decrescente
idx = S.argsort()[::-1]
S  = S[idx]
U  = U[:, idx]
Vt = Vt[idx, :]

# Scores dos filmes nos componentes principais: U * S
scores = U * S   # shape (n_filmes, K)

print(f"Valores singulares (top 10): {S[:10].round(2)}")
print(f"Shape dos scores: {scores.shape}")

## Scree plot — variância explicada por componente

In [ ]:
var_exp    = S**2 / (S**2).sum()
var_acum   = var_exp.cumsum()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(range(1, K + 1), var_exp * 100)
axes[0].set_xlabel('Componente principal')
axes[0].set_ylabel('Variância explicada (%)')
axes[0].set_title('Scree plot')

axes[1].plot(range(1, K + 1), var_acum * 100, marker='o', markersize=3)
axes[1].axhline(80, color='red', linestyle='--', label='80%')
axes[1].set_xlabel('Número de componentes')
axes[1].set_ylabel('Variância acumulada (%)')
axes[1].set_title('Variância explicada acumulada')
axes[1].legend()

plt.tight_layout()
plt.show()

n80 = int(np.searchsorted(var_acum, 0.80)) + 1
print(f"Componentes necessários para 80% da variância: {n80}")

## Scatter dos 2 primeiros componentes principais

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(scores[:, 0], scores[:, 1], alpha=0.15, s=4)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('Filmes no espaço dos 2 primeiros componentes principais')
plt.show()

## Interpretação dos componentes: atores que mais contribuem

Os loadings em `Vt` mostram quais atores "definem" cada PC.

In [ ]:
ator_idx_rev = {j: n for n, j in ator_idx.items()}   # índice → nconst

# Lookup nome → nconst no dataframe de actors
# actors tem index = nconst (ou coluna primaryName dependendo do parquet)
try:
    nome_ator = actors['primaryName']
except KeyError:
    nome_ator = actors.iloc[:, 0]   # fallback: primeira coluna

def top_atores_pc(pc_idx, n=10):
    loading = Vt[pc_idx]
    top_pos = loading.argsort()[::-1][:n]
    top_neg = loading.argsort()[:n]
    print(f"\n--- PC{pc_idx + 1} ---")
    print(f"  Lado positivo (loading > 0):")
    for j in top_pos:
        nc = ator_idx_rev[j]
        nome = nome_ator.get(nc, nc) if hasattr(nome_ator, 'get') else nc
        print(f"    {nome:30s}  loading={loading[j]:.4f}")
    print(f"  Lado negativo (loading < 0):")
    for j in top_neg:
        nc = ator_idx_rev[j]
        nome = nome_ator.get(nc, nc) if hasattr(nome_ator, 'get') else nc
        print(f"    {nome:30s}  loading={loading[j]:.4f}")

for pc in range(3):
    top_atores_pc(pc)

## Correlação com `averageRating`

Testamos se cada um dos primeiros componentes principais está correlacionado com a nota do filme, usando **Spearman** (sem assumir linearidade) e **teste de permutação** com amostra de 20.000 filmes.

H₀: não há correlação entre o score do PC e a nota do filme.

In [ ]:
def permutation_test_correlation(x, y, n_perm=10_000, rng=RNG):
    """
    Teste de permutação para correlação de Spearman.
    H0: não há correlação entre x e y (a ordem de y é irrelevante).
    Retorna o rho observado e o p-valor bicaudal.
    """
    x = np.asarray(x)
    y = np.asarray(y)

    observed_rho, _ = spearmanr(x, y)

    perm_rhos = np.empty(n_perm)
    for i in range(n_perm):
        shuffled_y = rng.permutation(y)
        perm_rhos[i], _ = spearmanr(x, shuffled_y)

    p_value = np.mean(np.abs(perm_rhos) >= np.abs(observed_rho))
    return observed_rho, p_value

In [ ]:
# Alinha ratings com a ordem de 'filmes' usada na matriz
ratings_indexed = ratings.set_index('tconst')['averageRating']

rating_vec = np.array([ratings_indexed.get(t, np.nan) for t in filmes])
valid_mask = ~np.isnan(rating_vec)

scores_valid = scores[valid_mask]
rating_valid = rating_vec[valid_mask]

PERM_SAMPLE_SIZE = 20_000
n_valid = valid_mask.sum()
print(f"Filmes com rating disponível: {n_valid:,}")

sample_idx = RNG.choice(n_valid, size=min(PERM_SAMPLE_SIZE, n_valid), replace=False)

N_PCS_TESTE = 5  # testar os primeiros 5 componentes

print("\n" + "="*65)
print(f"CORRELAÇÃO DOS {N_PCS_TESTE} PRIMEIROS PCs COM averageRating")
print("="*65)
print(f"{'PC':<6} {'rho (completo)':>16} {'p scipy':>12} {'rho (amostra)':>15} {'p perm':>10}")
print("-"*65)

for pc in range(N_PCS_TESTE):
    pc_scores_full   = scores_valid[:, pc]
    pc_scores_sample = pc_scores_full[sample_idx]
    rating_sample    = rating_valid[sample_idx]

    rho_full, p_full = spearmanr(pc_scores_full, rating_valid)
    rho_perm, p_perm = permutation_test_correlation(pc_scores_sample, rating_sample)

    print(f"PC{pc+1:<4} {rho_full:>16.4f} {p_full:>12.5f} {rho_perm:>15.4f} {p_perm:>10.5f}")

print("-"*65)
print(f"Tamanho da amostra para permutação: {len(sample_idx):,}  (10.000 permutações)")